## Imports

In [0]:
from pyspark.sql.functions import col, current_timestamp

## Dynamic Catalog

In [0]:
# Catalog parameter
dbutils.widgets.text("CATALOG", "vp_vk_mkc20040528")
CATALOG = dbutils.widgets.get("CATALOG")

## Base Path

In [0]:
BASE_PATH = f"/Volumes/{CATALOG}/landing/raw"

## Stations ingestion

In [0]:
# Stations ingestion
stations_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{BASE_PATH}/stations/")
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file"),
        current_timestamp().alias("_ingested_at")
    )
)

stations_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.stations")

## Tariffs ingestion

In [0]:
# Tariffs ingestion
tariffs_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{BASE_PATH}/tariffs/")
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file"),
        current_timestamp().alias("_ingested_at")
    )
)

tariffs_df.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.tariffs")

In [0]:
%sql
select * from vp_vk_mkc20040528.bronze.stations limit 10

In [0]:
%sql
select * from vp_vk_mkc20040528.bronze.tariffs limit 10

## Members ingestion

In [0]:
%sql
DESCRIBE TABLE vp_vk_mkc20040528.bronze.members;

In [0]:
%sql
-- Create members Bronze table
CREATE TABLE IF NOT EXISTS IDENTIFIER(:CATALOG).bronze.members
(
    member_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    plan STRING,
    joined_on STRING,
    change_effective_date STRING,
    _source_file STRING,
    _ingested_at TIMESTAMP
);

-- Load both snapshot and change files with metadata
COPY INTO IDENTIFIER(:CATALOG).bronze.members
FROM (
    SELECT
        *,
        _metadata.file_path AS _source_file,
        current_timestamp() AS _ingested_at
    FROM '/Volumes/' :CATALOG '/landing/raw/members/'
)
FILEFORMAT = CSV
FORMAT_OPTIONS (
    'header' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
%sql
select * from vp_vk_mkc20040528.bronze.members limit 10

## Chargers ingestion

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:CATALOG).bronze.chargers;

In [0]:
%sql
COPY INTO IDENTIFIER(:CATALOG).bronze.chargers
FROM (
  SELECT
    *,
    _metadata.file_path AS _source_file,
    current_timestamp() AS _ingested_at
  FROM '/Volumes/' :CATALOG '/landing/raw/chargers/'
)
FILEFORMAT = JSON
FORMAT_OPTIONS (
    'header' = 'true',
    'inferSchema' = 'false',
    'multiLine' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
);

In [0]:
%sql
SELECT * FROM vp_vk_mkc20040528.bronze.chargers LIMIT 10